# Chapter 2: The Supply Chain Graph

This notebook builds the static supply chain network in Neo4j Aura. We generate a synthetic network of suppliers, warehouses, distribution centers and retailers, load it into Neo4j and demonstrate shortest path queries and alternative routing after a disruption.

This is the graph layer only. No shipment data is loaded here -- shipment events arrive via Confluent Kafka in Chapter 5 and land on the nodes we create now.

## What We Build

```
Suppliers -> Warehouses -> Distribution Centers -> Retailers
   (20)         (12)              (10)               (30)
```

Each arrow represents a `SHIPS_TO` relationship carrying three properties:

- `cost` -- shipping cost in dollars
- `distance` -- distance in kilometres
- `capacity` -- maximum units per shipment

## 1. Install Dependencies

All versions are pinned for reproducibility. Run this cell once.

In [1]:
%pip install faker==40.36.0 \
             neo4j==6.2.0 \
             numpy==2.2.6 \
             pandas==3.0.5 --quiet

print("Install complete.")

Note: you may need to restart the kernel to use updated packages.
Install complete.


## 2. Imports

In [2]:
import os
import numpy as np
import pandas as pd

from faker import Faker
from neo4j import GraphDatabase

## 3. Configuration

Connection details are read from environment variables. Node counts and the random seed match the Kafka producer and consumer notebooks so shipment events in Chapter 5 land on nodes that already exist in the graph.

In [3]:
NEO4J_URI      = os.environ["NEO4J_URI"]
NEO4J_USERNAME = os.environ["NEO4J_USERNAME"]
NEO4J_PASSWORD = os.environ["NEO4J_PASSWORD"]

N_SUPPLIERS    = 20
N_WAREHOUSES   = 12
N_DIST_CENTERS = 10
N_RETAILERS    = 30
RANDOM_SEED    = 42

print("Configuration set.")

Configuration set.


## 4. Connect to Neo4j

We verify connectivity before doing any work. `verify_connectivity()` returns `None` on success -- this is expected.

In [4]:
driver = GraphDatabase.driver(
    NEO4J_URI,
    auth = (NEO4J_USERNAME, NEO4J_PASSWORD),
    notifications_min_severity = "OFF",
)

print(driver.verify_connectivity()) # None is expected
print("Connected to Neo4j Aura.")

None
Connected to Neo4j Aura.


## 5. Clear Existing Data

We delete all existing nodes and relationships so each run starts from a clean graph. We use `CALL { ... } IN TRANSACTIONS` to avoid memory issues on larger graphs.

In [5]:
with driver.session() as session:
    session.run("""
        MATCH (n)
        CALL (n) { DETACH DELETE n }
        IN TRANSACTIONS OF 10000 ROWS
    """)

print("Graph cleared.")

Graph cleared.


## 6. Generate the Supply Chain Network

We use Faker and NumPy to generate a realistic but entirely synthetic supply chain. A fixed random seed means the dataset is identical on every run and node IDs match the Kafka producer in Chapter 5.

The `make_routes()` function guarantees minimum connectivity: every source node gets at least two outbound routes and every target node gets at least two inbound routes. This ensures the graph stays connected after a single node disruption.

In [6]:
fake = Faker()
Faker.seed(RANDOM_SEED)
rng = np.random.default_rng(RANDOM_SEED)

REGIONS = ["North", "South", "East", "West", "Central"]

def make_suppliers(n):
    return pd.DataFrame([{
        "id":          f"S{i:03d}",
        "name":        fake.company(),
        "region":      str(rng.choice(REGIONS)),
        "capacity":    int(rng.integers(500, 2000)),
        "reliability": round(float(rng.uniform(0.7, 1.0)), 2),
    } for i in range(n)])

def make_warehouses(n):
    return pd.DataFrame([{
        "id":       f"W{i:03d}",
        "name":     f"{fake.city()} Warehouse",
        "region":   str(rng.choice(REGIONS)),
        "capacity": int(rng.integers(1000, 5000)),
    } for i in range(n)])

def make_dist_centers(n):
    return pd.DataFrame([{
        "id":     f"DC{i:03d}",
        "name":   f"{fake.city()} Distribution Center",
        "region": str(rng.choice(REGIONS)),
    } for i in range(n)])

def make_retailers(n):
    return pd.DataFrame([{
        "id":     f"R{i:03d}",
        "name":   fake.company(),
        "region": str(rng.choice(REGIONS)),
        "demand": int(rng.integers(100, 800)),
    } for i in range(n)])

def make_routes(sources, targets, n_routes, min_out=2, min_in=2):
    """Generate directed routes with guaranteed minimum connectivity.

    Every source gets at least min_out outbound routes.
    Every target gets at least min_in inbound routes.
    This ensures no single node removal can fully disconnect the network.
    """
    rows, seen = [], set()
    src_ids = sources["id"].tolist()
    tgt_ids = targets["id"].tolist()

    def make_row(s, t):
        seen.add((s, t))
        return {
            "source":   s,
            "target":   t,
            "cost":     round(float(rng.uniform(10, 500)), 2),
            "distance": int(rng.integers(50, 2000)),
            "capacity": int(rng.integers(200, 1000)),
        }

    # Guarantee every source has at least min_out outbound routes
    for s in src_ids:
        sample = rng.choice(tgt_ids, size=min(min_out, len(tgt_ids)), replace=False)
        for t in sample:
            t = str(t)
            if (s, t) not in seen:
                rows.append(make_row(s, t))

    # Guarantee every target has at least min_in inbound routes
    for t in tgt_ids:
        sample = rng.choice(src_ids, size=min(min_in, len(src_ids)), replace=False)
        for s in sample:
            s = str(s)
            if (s, t) not in seen:
                rows.append(make_row(s, t))

    # Fill remaining routes randomly up to n_routes
    attempts = 0
    while len(rows) < n_routes and attempts < n_routes * 10:
        attempts += 1
        s = str(rng.choice(src_ids))
        t = str(rng.choice(tgt_ids))
        if (s, t) not in seen:
            rows.append(make_row(s, t))

    return pd.DataFrame(rows)

suppliers    = make_suppliers(N_SUPPLIERS)
warehouses   = make_warehouses(N_WAREHOUSES)
dist_centers = make_dist_centers(N_DIST_CENTERS)
retailers    = make_retailers(N_RETAILERS)
sup_to_wh    = make_routes(suppliers,    warehouses,   35)
wh_to_dc     = make_routes(warehouses,   dist_centers, 30)
dc_to_ret    = make_routes(dist_centers, retailers,    60)

print("Dataset summary")
print(f"  Suppliers             : {len(suppliers)}")
print(f"  Warehouses            : {len(warehouses)}")
print(f"  Distribution Centers  : {len(dist_centers)}")
print(f"  Retailers             : {len(retailers)}")
print(f"  Supplier -> Warehouse : {len(sup_to_wh)} routes")
print(f"  Warehouse -> DC       : {len(wh_to_dc)} routes")
print(f"  DC -> Retailer        : {len(dc_to_ret)} routes")
print(f"  Total nodes           : {len(suppliers)+len(warehouses)+len(dist_centers)+len(retailers)}")
print(f"  Total routes          : {len(sup_to_wh)+len(wh_to_dc)+len(dc_to_ret)}")

Dataset summary
  Suppliers             : 20
  Warehouses            : 12
  Distribution Centers  : 10
  Retailers             : 30
  Supplier -> Warehouse : 61 routes
  Warehouse -> DC       : 38 routes
  DC -> Retailer        : 73 routes
  Total nodes           : 72
  Total routes          : 172


## 7. Load Nodes into Neo4j

We load each node type in a separate batch using `UNWIND`. `MERGE` on the `id` property is idempotent -- re-running this cell does not create duplicate nodes.

In [7]:
def load_nodes(session, label, df):
    """Load a DataFrame of nodes into Neo4j using UNWIND."""
    session.run(
        f"""
        UNWIND $rows AS row
        MERGE (n:{label} {{id: row.id}})
        SET n += row
        """,
        rows=df.to_dict(orient="records")
    )

with driver.session() as session:
    load_nodes(session, "Supplier",           suppliers)
    load_nodes(session, "Warehouse",          warehouses)
    load_nodes(session, "DistributionCenter", dist_centers)
    load_nodes(session, "Retailer",           retailers)

print("Nodes loaded.")

Nodes loaded.


## 8. Load Relationships into Neo4j

We load each relationship tier separately. `MERGE` on the combination of source, target and relationship type prevents duplicate routes on re-runs. Route properties (`cost`, `distance`, `capacity`) are set with `SET r += row`.

In [8]:
def load_routes(session, src_label, tgt_label, df):
    """Load a DataFrame of routes as SHIPS_TO relationships."""
    session.run(
        f"""
        UNWIND $rows AS row
        MATCH (src:{src_label} {{id: row.source}})
        MATCH (tgt:{tgt_label} {{id: row.target}})
        MERGE (src)-[r:SHIPS_TO]->(tgt)
        SET r.cost     = row.cost,
            r.distance = row.distance,
            r.capacity = row.capacity
        """,
        rows=df.to_dict(orient="records")
    )

with driver.session() as session:
    load_routes(session, "Supplier",           "Warehouse",          sup_to_wh)
    load_routes(session, "Warehouse",          "DistributionCenter", wh_to_dc)
    load_routes(session, "DistributionCenter", "Retailer",           dc_to_ret)

print("Relationships loaded.")

Relationships loaded.


## 9. Create Indexes

We create an index on the `id` property for each node label. The Kafka consumer in Chapter 5 looks up nodes by `id` on every event -- these indexes make those lookups fast.

In [9]:
with driver.session() as session:
    for label in ["Supplier", "Warehouse", "DistributionCenter", "Retailer"]:
        session.run(
            f"CREATE INDEX {label.lower()}_id IF NOT EXISTS "
            f"FOR (n:{label}) ON (n.id)"
        )
    session.run(
        "CREATE INDEX shipment_status IF NOT EXISTS "
        "FOR (n:Shipment) ON (n.status)"
    )

print("Indexes created.")

Indexes created.


## 10. Verify the Graph

We run a summary query to confirm the node and relationship counts match what we loaded.

In [10]:
with driver.session() as session:
    result = session.run("""
        MATCH (n)
        RETURN labels(n)[0] AS label, count(n) AS count
        ORDER BY label
    """)
    print("Node counts:")
    for row in result:
        print(f"  {row['label']:<22} : {row['count']}")

    result = session.run("""
        MATCH ()-[r:SHIPS_TO]->()
        RETURN count(r) AS total_routes
    """)
    print(f"\nTotal SHIPS_TO routes    : {result.single()['total_routes']}")

Node counts:
  DistributionCenter     : 10
  Retailer               : 30
  Supplier               : 20
  Warehouse              : 12

Total SHIPS_TO routes    : 172


## 11. Shortest Path Query

With the network in place we can ask Neo4j to find the shortest path from a supplier to a retailer across the full supply chain. `shortestPath()` uses a BFS traversal and returns the fewest-hop path.

We pick a supplier and retailer as a concrete example. Change `SUPPLIER_ID` and `RETAILER_ID` to explore other paths.

In [11]:
SUPPLIER_ID = "S000"
RETAILER_ID = "R001"

with driver.session() as session:
    result = session.run("""
        MATCH (s:Supplier {id: $supplier_id}),
              (r:Retailer {id: $retailer_id})
        MATCH path = shortestPath((s)-[:SHIPS_TO*]->(r))
        RETURN
            [n IN nodes(path) | coalesce(n.id, '?')] AS node_ids,
            [n IN nodes(path) | labels(n)[0]]        AS node_labels,
            length(path)                             AS hops,
            round(reduce(
                total = 0.0, r IN relationships(path) | total + r.cost
            ), 2)                                    AS total_cost,
            round(reduce(
                total = 0.0, r IN relationships(path) | total + r.distance
            ), 0)                                    AS total_distance_km
    """,
        supplier_id=SUPPLIER_ID,
        retailer_id=RETAILER_ID,
    )

    record = result.single()
    if record:
        print(f"Shortest path: {SUPPLIER_ID} -> {RETAILER_ID}")
        print(f"  Hops         : {record['hops']}")
        print(f"  Total cost   : ${record['total_cost']}")
        print(f"  Total dist.  : {int(record['total_distance_km'])} km")
        print()
        print("  Route:")
        for nid, nlabel in zip(record["node_ids"], record["node_labels"]):
            print(f"    {nlabel:<22} {nid}")
    else:
        print(f"No path found between {SUPPLIER_ID} and {RETAILER_ID}.")

Shortest path: S000 -> R001
  Hops         : 3
  Total cost   : $744.73
  Total dist.  : 2750 km

  Route:
    Supplier               S000
    Warehouse              W005
    DistributionCenter     DC005
    Retailer               R001


## 12. Disruption and Alternative Routing

This is where graph routing earns its place. We mark a warehouse as inactive to simulate a disruption -- a flood, a strike, a capacity failure -- then ask Neo4j to find the shortest path that avoids the disrupted node.

The same query pattern is used in Chapter 6 when the simulator fires a live disruption mid-stream and the Kafka consumer needs to reroute in real time.

In [12]:
DISRUPTED_WAREHOUSE = "W003"

# Mark the warehouse as inactive
with driver.session() as session:
    result = session.run(
        "MATCH (w:Warehouse {id: $id}) SET w.active = false RETURN w.id AS id",
        id=DISRUPTED_WAREHOUSE
    )
    row = result.single()
    if row:
        print(f"Warehouse {row['id']} marked as inactive.")
    else:
        print(f"WARNING: Warehouse {DISRUPTED_WAREHOUSE} not found.")

Warehouse W003 marked as inactive.


In [13]:
with driver.session() as session:
    result = session.run("""
        MATCH (s:Supplier {id: $supplier_id}),
              (r:Retailer {id: $retailer_id})
        MATCH path = (s)-[:SHIPS_TO*1..6]->(r)
        WHERE NONE(
            n IN nodes(path)
            WHERE n:Warehouse AND coalesce(n.active, true) = false
        )
        RETURN
            [n IN nodes(path) | coalesce(n.id, '?')] AS node_ids,
            [n IN nodes(path) | labels(n)[0]]        AS node_labels,
            length(path)                             AS hops,
            round(reduce(
                total = 0.0, r IN relationships(path) | total + r.cost
            ), 2)                                    AS total_cost,
            round(reduce(
                total = 0.0, r IN relationships(path) | total + r.distance
            ), 0)                                    AS total_distance_km
        ORDER BY hops ASC
        LIMIT 1
    """,
        supplier_id=SUPPLIER_ID,
        retailer_id=RETAILER_ID,
    )

    record = result.single()
    if record:
        print(f"Alternative path avoiding {DISRUPTED_WAREHOUSE}: {SUPPLIER_ID} -> {RETAILER_ID}")
        print(f"  Hops         : {record['hops']}")
        print(f"  Total cost   : ${record['total_cost']}")
        print(f"  Total dist.  : {int(record['total_distance_km'])} km")
        print()
        print("  Route:")
        for nid, nlabel in zip(record["node_ids"], record["node_labels"]):
            print(f"    {nlabel:<22} {nid}")
    else:
        print(
            f"No alternative path found. The network may not have enough redundancy "
            f"to route around {DISRUPTED_WAREHOUSE} for this supplier-retailer pair."
        )

Alternative path avoiding W003: S000 -> R001
  Hops         : 3
  Total cost   : $744.73
  Total dist.  : 2750 km

  Route:
    Supplier               S000
    Warehouse              W005
    DistributionCenter     DC005
    Retailer               R001


In [14]:
# Restore the warehouse to active so the graph is clean for Chapter 5
with driver.session() as session:
    session.run(
        "MATCH (w:Warehouse {id: $id}) REMOVE w.active",
        id=DISRUPTED_WAREHOUSE
    )

print(f"Warehouse {DISRUPTED_WAREHOUSE} restored to active.")

Warehouse W003 restored to active.


## 13. Teardown

Close the Neo4j driver. The graph data stays in Aura -- we need it in Chapter 5 when the Kafka consumer starts writing shipment events.

In [15]:
driver.close()
print("Neo4j connection closed.")

Neo4j connection closed.
